# Stage 2: Data loading and understanding
Load the eight Home Credit tables, inspect their size and grain, verify how they join, and build a
data dictionary we can reuse throughout the project.

**How the tables fit together**
```
                       application_{train|test}   (1 row per current application, SK_ID_CURR)
                        /                       \
             bureau (SK_ID_CURR)            previous_application (SK_ID_CURR)
   credits at OTHER lenders                 earlier applications at Home Credit (SK_ID_PREV)
               |                              /            |                 \
     bureau_balance (SK_ID_BUREAU)   POS_CASH_balance  installments_payments  credit_card_balance
     monthly status of each          monthly loan      every repayment made   monthly card
     bureau credit                   snapshots         (or missed)            snapshots
```
Every child table is *one-to-many*: before modelling, each needs to be aggregated to one row per `SK_ID_CURR`.

In [1]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd

from credit_risk.config import get_paths, load_config
from credit_risk.data_loader import (
    TABLE_INFO,
    build_data_dictionary,
    check_relationships,
    load_all_tables,
    load_column_descriptions,
    table_overview,
)

cfg = load_config()
paths = get_paths(cfg)
pd.set_option("display.max_columns", 50)

## 1. Load every table
Numeric columns are downcast (float64 → float32, smallest int type) and repetitive text columns of the
large tables are read as `category`. On the full data this cuts memory from ~10 GB to ~4 GB.
Set `SAMPLE_ROWS` to e.g. `200_000` on a low-memory laptop.

In [2]:
SAMPLE_ROWS = None

tables = load_all_tables(cfg, nrows=SAMPLE_ROWS)
overview = table_overview(tables)
overview

,table,rows,columns,memory_mb,cols_with_missing,avg_missing_pct,grain
0,application_train,307511,122,167.1,67,24.4,One row per current loan application (with TAR...
1,application_test,48744,121,26.4,64,23.8,One row per current loan application (no TARGE...
2,bureau,1716428,17,85.1,7,13.5,Client's previous credits at OTHER institution...
3,bureau_balance,27299925,3,156.2,0,0.0,Monthly status of each bureau credit. Grain: S...
4,previous_application,1670214,37,146.5,16,18.0,Previous applications to Home Credit. Grain: S...
5,pos_cash_balance,10001358,8,209.8,2,0.1,Monthly snapshots of previous POS / cash loans...
6,installments_payments,13605401,8,389.3,2,0.0,Repayment history of previous Home Credit loan...
7,credit_card_balance,3840312,23,293.0,9,6.7,Monthly snapshots of previous Home Credit cred...


## 2. A first look at each table

In [3]:
for name, df in tables.items():
    print(f"\n=== {name}  {df.shape[0]:,} rows x {df.shape[1]} cols ===\n{TABLE_INFO[name]}")
    display(df.head(3))


=== application_train  307,511 rows x 122 cols ===
One row per current loan application (with TARGET). Grain: SK_ID_CURR.


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,...,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0



=== application_test  48,744 rows x 121 cols ===
One row per current loan application (no TARGET, Kaggle scoring set).


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,...,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,Unaccompanied,Working,Higher education,Married,House / apartment,0.018850,-19241,-2329,-5170.0,-812,NaN,1,1,0,1,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,100005,Cash loans,M,N,Y,0,99000.0,222768.0,17370.0,180000.0,Unaccompanied,Working,Secondary / secondary special,Married,House / apartment,0.035792,-18064,-4469,-9118.0,-1623,NaN,1,1,0,1,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
2,100013,Cash loans,M,Y,Y,0,202500.0,663264.0,69777.0,630000.0,NaN,Working,Higher education,Married,House / apartment,0.019101,-20038,-4458,-2175.0,-3503,5.0,1,1,0,1,...,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0,4.0



=== bureau  1,716,428 rows x 17 cols ===
Client's previous credits at OTHER institutions reported to the Credit Bureau. Grain: SK_ID_BUREAU.


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN


... (6 outputs left out)


=== installments_payments  13,605,401 rows x 8 cols ===
Repayment history of previous Home Credit loans. Grain: one row per payment.


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.359863,6948.359863
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525024,1716.525024
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000000,25425.000000



=== credit_card_balance  3,840,312 rows x 23 cols ===
Monthly snapshots of previous Home Credit credit cards. Grain: SK_ID_PREV x MONTHS_BALANCE.


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,AMT_PAYMENT_CURRENT,AMT_PAYMENT_TOTAL_CURRENT,AMT_RECEIVABLE_PRINCIPAL,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970001,135000,0.0,877.5,0.0,877.5,1700.324951,1800.0,1800.0,0.000000,0.000000,0.000000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.554688,45000,2250.0,2250.0,0.0,0.0,2250.000000,2250.0,2250.0,60175.078125,64875.554688,64875.554688,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.224609,450000,0.0,0.0,0.0,0.0,2250.000000,2250.0,2250.0,26926.425781,31460.085938,31460.085938,0.0,0,0.0,0.0,30.0,Active,0,0


## 3. Verify the relationships
- `child_keys_matching_parent_pct` below 100% for `application_train → bureau` is **expected**: the history tables
  also cover clients in `application_test`.
- `parents_with_children_pct` shows how many applicants actually *have* a history in that table. Clients
  with no bureau or previous-application record are "thin-file" customers, a known higher-risk segment.

In [4]:
rels = check_relationships(tables)
rels

,parent,child,key,cardinality,child_rows,child_keys_matching_parent_pct,parents_with_children_pct,avg_children_per_parent
0,application_train,bureau,SK_ID_CURR,1 : 0..N,1716428,86.2,85.7,5.6
1,bureau,bureau_balance,SK_ID_BUREAU,1 : 0..N,27299925,94.7,45.1,33.4
2,application_train,previous_application,SK_ID_CURR,1 : 0..N,1670214,85.9,94.6,4.9
3,previous_application,pos_cash_balance,SK_ID_PREV,1 : 0..N,10001358,96.0,53.8,10.7
4,previous_application,installments_payments,SK_ID_PREV,1 : 0..N,13605401,96.1,57.4,13.6
5,previous_application,credit_card_balance,SK_ID_PREV,1 : 0..N,3840312,89.1,5.6,36.8


In [5]:
app_ids = pd.Index(pd.concat([tables["application_train"]["SK_ID_CURR"], tables["application_test"]["SK_ID_CURR"]]))
coverage = {
    name: round(100 * app_ids.isin(tables[name]["SK_ID_CURR"].unique()).mean(), 1)
    for name in ["bureau", "previous_application", "pos_cash_balance", "installments_payments", "credit_card_balance"]
}
pd.Series(coverage, name="% of applicants with records").to_frame()

,% of applicants with records
bureau,85.8
previous_application,95.1
pos_cash_balance,94.7
installments_payments,95.3
credit_card_balance,29.1


## 4. Target variable at a glance
`TARGET = 1` means the client had payment difficulties. It is only available in `application_train`.

In [6]:
t = tables["application_train"]["TARGET"].value_counts().rename(index={0: "Repaid (0)", 1: "Payment difficulties (1)"})
display(pd.DataFrame({"applicants": t, "share_pct": (100 * t / t.sum()).round(2)}))
print(f"Imbalance: {t.iloc[0] / t.iloc[1]:.1f} good borrowers per defaulter")

,applicants,share_pct
TARGET,,
Repaid (0),282686,91.93
Payment difficulties (1),24825,8.07


Imbalance: 11.4 good borrowers per defaulter


## 5. Data dictionary
Combines Home Credit's official column descriptions with profiling stats (type, % missing, cardinality, range).
It is saved to `reports/data_dictionary.csv` and is the first thing a reviewer or new team member should read.

In [7]:
descriptions = load_column_descriptions(cfg)
dictionary = build_data_dictionary(tables, descriptions)
out = paths["reports"] / "data_dictionary.csv"
dictionary.to_csv(out, index=False)
overview.to_csv(paths["reports"] / "table_overview.csv", index=False)
print(f"Saved {len(dictionary):,} column entries -> {out.relative_to(ROOT)}")
dictionary[dictionary["table"] == "application_train"].head(25)

Saved 339 column entries -> reports\data_dictionary.csv


,table,column,dtype,kind,missing_pct,n_unique,example,min,max,description,notes
0,application_train,SK_ID_CURR,int32,numeric,0.00,307511,100002,100002.00000,4.562550e+05,ID of loan in our sample,NaN
1,application_train,TARGET,int8,numeric,0.00,2,1,0.00000,1.000000e+00,Target variable (1 - client with payment diffi...,NaN
2,application_train,NAME_CONTRACT_TYPE,str,categorical,0.00,2,Cash loans,NaN,NaN,Identification if loan is cash or revolving,NaN
3,application_train,CODE_GENDER,str,categorical,0.00,3,M,NaN,NaN,Gender of the client,NaN
4,application_train,FLAG_OWN_CAR,str,categorical,0.00,2,N,NaN,NaN,Flag if the client owns a car,NaN
5,application_train,FLAG_OWN_REALTY,str,categorical,0.00,2,Y,NaN,NaN,Flag if client owns a house or flat,NaN
6,application_train,CNT_CHILDREN,int8,numeric,0.00,15,0,0.00000,1.900000e+01,Number of children the client has,NaN
7,application_train,AMT_INCOME_TOTAL,float32,numeric,0.00,2548,202500.0,25650.00000,1.170000e+08,Income of the client,NaN
8,application_train,AMT_CREDIT,float32,numeric,0.00,5603,406597.5,45000.00000,4.050000e+06,Credit amount of the loan,NaN
9,application_train,AMT_ANNUITY,float32,numeric,0.00,13672,24700.5,1615.50000,2.580255e+05,Loan annuity,NaN


In [8]:
# Columns grouped by theme (application table): useful when explaining the data in an interview
app_cols = tables["application_train"].columns
themes = {
    "Loan request": [c for c in app_cols if c.startswith(("AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS", "NAME_CONTRACT"))],
    "Income & employment": [c for c in app_cols if c in ("AMT_INCOME_TOTAL", "NAME_INCOME_TYPE", "DAYS_EMPLOYED", "OCCUPATION_TYPE", "ORGANIZATION_TYPE")],
    "Demographics": [c for c in app_cols if c in ("CODE_GENDER", "DAYS_BIRTH", "NAME_FAMILY_STATUS", "CNT_CHILDREN", "CNT_FAM_MEMBERS", "NAME_EDUCATION_TYPE")],
    "External scores": [c for c in app_cols if c.startswith("EXT_SOURCE")],
    "Housing / building stats": [c for c in app_cols if c.endswith(("_AVG", "_MODE", "_MEDI")) or c == "NAME_HOUSING_TYPE"],
    "Documents provided": [c for c in app_cols if c.startswith("FLAG_DOCUMENT")],
    "Bureau enquiries": [c for c in app_cols if c.startswith("AMT_REQ_CREDIT_BUREAU")],
    "Social circle": [c for c in app_cols if "SOCIAL_CIRCLE" in c],
    "Region & process": [c for c in app_cols if c.startswith(("REGION_", "REG_", "LIVE_", "WEEKDAY_", "HOUR_"))],
}
pd.DataFrame({"n_columns": {k: len(v) for k, v in themes.items()}, "examples": {k: ", ".join(v[:4]) for k, v in themes.items()}})

,n_columns,examples
Loan request,4,"NAME_CONTRACT_TYPE, AMT_CREDIT, AMT_ANNUITY, A..."
Income & employment,5,"AMT_INCOME_TOTAL, NAME_INCOME_TYPE, DAYS_EMPLO..."
Demographics,6,"CODE_GENDER, CNT_CHILDREN, NAME_EDUCATION_TYPE..."
External scores,3,"EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3"
Housing / building stats,48,"NAME_HOUSING_TYPE, APARTMENTS_AVG, BASEMENTARE..."
Documents provided,20,"FLAG_DOCUMENT_2, FLAG_DOCUMENT_3, FLAG_DOCUMEN..."
Bureau enquiries,6,"AMT_REQ_CREDIT_BUREAU_HOUR, AMT_REQ_CREDIT_BUR..."
Social circle,4,"OBS_30_CNT_SOCIAL_CIRCLE, DEF_30_CNT_SOCIAL_CI..."
Region & process,11,"REGION_POPULATION_RELATIVE, REGION_RATING_CLIE..."


## Key takeaways
- **Grain matters:** only the application table is one-row-per-applicant. Everything else must be aggregated,
  which is where most of the predictive signal (payment behaviour) is created in Stage 6.
- **Time is relative:** all `DAYS_*` / `MONTHS_BALANCE` columns count backwards from the application date
  (negative = in the past). That makes it simple to guarantee we only use information available at decision time.
- **Imbalanced target:** roughly 1 defaulter for every 11 good borrowers on the full data, so accuracy will be misleading.